# Milestone 2: RAG Retrieval Implementations
This notebook houses the `VectorRetriever` and `GraphRetriever` classes that ingest the generated Chroma and NetworkX knowledge bases, respectively. It demonstrates how queries are routed and how historical contexts are fetched.

In [ ]:
!pip install -q chromadb sentence-transformers networkx

In [ ]:
import os
import pickle
import chromadb
import networkx as nx
from typing import List, Dict, Any
from sentence_transformers import SentenceTransformer

class VectorRetriever:
    """ Retrieves similar evidence clauses from the historical knowledge base using Vector search. """
    def __init__(self, index_path: str, collection_name: str = "m2_training_rag"):
        self.index_path = index_path
        print(f"[VectorRetriever] Connecting to ChromaDB at {index_path}")
        self.client = chromadb.PersistentClient(path=index_path)
        self.collection = self.client.get_collection(name=collection_name)
        # Load the same lightweight embedding model on CPU
        self.embed_model = SentenceTransformer('all-MiniLM-L6-v2', device='cpu')
        
    def retrieve(self, query: str, top_k: int = 3) -> List[Dict[str, Any]]:
        print(f"[VectorRetriever] Searching for: '{query}'")
        query_embedding = self.embed_model.encode([query]).tolist()
        
        results = self.collection.query(
            query_embeddings=query_embedding,
            n_results=top_k
        )
        
        retrieved_contexts = []
        if results and results['documents']:
            for doc, meta in zip(results['documents'][0], results['metadatas'][0]):
                retrieved_contexts.append({
                    "source": "vector_db",
                    "text": doc,
                    "metadata": meta
                })
                
        return retrieved_contexts


class GraphRetriever:
    """ Retrieves relevant interconnected rules and clauses using a Graph Knowledge Base. """
    def __init__(self, graph_path: str):
        self.graph_path = graph_path
        print(f"[GraphRetriever] Loading NetworkX graph from {graph_path}")
        with open(graph_path, "rb") as f:
            self.G = pickle.load(f)
            
        # Use embeddings to find the best entry nodes in the graph
        self.embed_model = SentenceTransformer('all-MiniLM-L6-v2', device='cpu')
        
        # Pre-compute embeddings for all Rule (Hypothesis) nodes for semantic entry
        self.rule_nodes = [n for n, attr in self.G.nodes(data=True) if attr.get('type') == 'Hypothesis']
        self.rule_texts = [self.G.nodes[n].get('text', '') for n in self.rule_nodes]
        if self.rule_texts:
            self.rule_embeddings = self.embed_model.encode(self.rule_texts)
        
    def retrieve(self, query: str, top_k_rules: int = 2) -> List[Dict[str, Any]]:
        print(f"[GraphRetriever] Traversing graph for: '{query}'")
        if not self.rule_nodes:
            return []
            
        query_emb = self.embed_model.encode(query)
        
        # Find the most similar semantic Rule/Hypothesis nodes
        from sentence_transformers import util
        similarities = util.cos_sim(query_emb, self.rule_embeddings)[0]
        top_indices = similarities.topk(k=min(top_k_rules, len(self.rule_nodes))).indices.tolist()
        
        retrieved_contexts = []
        
        # Traverse the graph: Given the relevant Rule, find Clauses connected to it
        for idx in top_indices:
            rule_node = self.rule_nodes[idx]
            rule_text = self.rule_texts[idx]
            
            # Find incoming edges (Clauses that entail/contradict this Rule)
            connected_clauses = list(self.G.predecessors(rule_node))
            
            # Just take up to 20 connected clauses for brevity
            # TODO: Choose number of clauses to retrieve
            num_clauses_to_retrieve = min(20, len(connected_clauses))
            for clause_node in connected_clauses[:num_clauses_to_retrieve]:
                clause_data = self.G.nodes[clause_node]
                edge_data = self.G.get_edge_data(clause_node, rule_node)
                relation = edge_data.get('relation', 'Unknown')
                
                chunk_text = (
                    f"Graph Traversal -> Rule: '{rule_text}'\n"
                    f"Historical Outcome: {relation}\n"
                    f"Supporting Clause: '{clause_data.get('text', '')}'"
                )
                
                retrieved_contexts.append({
                    "source": "graph_db",
                    "text": chunk_text,
                    "metadata": {"rule": rule_node, "clause": clause_node, "relation": relation}
                })
                
        return retrieved_contexts

### Test Retrievers locally
Note: To run this test, you must have successfully generated the indices in step 1 and 2.

In [ ]:
# Test Vector Retriever
if os.path.exists("/kaggle/working/vector_db"):
    vr = VectorRetriever(index_path="/kaggle/working/vector_db")
    res_v = vr.retrieve("Does this agreement survive after termination?", top_k=2)
    print("\n--- Vector Results ---")
    for r in res_v:
        print(r['text'])
        print("-" * 20)
else:
    print("Vector DB not found at /kaggle/working/vector_db")

# Test Graph Retriever
graph_file = "/kaggle/working/graph_db/contractnli_graph.pkl"
if os.path.exists(graph_file):
    gr = GraphRetriever(graph_path=graph_file)
    res_g = gr.retrieve("Does this agreement survive after termination?", top_k_rules=1)
    print("\n--- Graph Results ---")
    for r in res_g:
        print(r['text'])
        print("-" * 20)
else:
    print(f"Graph DB not found at {graph_file}")

### Conversational Agent
This section implements a stateful conversational agent using HuggingFace `transformers` and loads `Qwen/Qwen2.5-3B-Instruct` using 4-bit quantization, similar to the setup from Milestone 1.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

class ConversationalAgent:
    """ Stateful Conversation Layer incorporating RAG context for Qwen2.5-3B-Instruct. """
    def __init__(self, model_identifier="Qwen/Qwen2.5-3B-Instruct"):
        self.model_identifier = model_identifier
        self.history = []
        
        # Load model using QLoRA config from Milestone 1
        print(f"[Agent] Loading {model_identifier} in 4-bit...")
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True, 
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16, 
            bnb_4bit_use_double_quant=True,
        )
        self.tokenizer = AutoTokenizer.from_pretrained(model_identifier, trust_remote_code=True)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
            
        self.model = AutoModelForCausalLM.from_pretrained(
            model_identifier, 
            quantization_config=bnb_config,
            device_map="auto", 
            torch_dtype=torch.float16,
            trust_remote_code=True
        )
        self.model.eval()
        print("[Agent] Model Loaded Successfully.")
        
    @torch.inference_mode()
    def generate_response(self, user_prompt: str, contract_text: str, context: str, max_new_tokens=256) -> str:
        """ Combines external RAG context with raw contract text and conversation history. """
        
        # We only instruct the system once globally
        # Enforcing Grounding Constraints from MS1
        system_message = (
            "You are an expert legal aide reviewing NDAs. "
            "Use the provided External Context ONLY to understand how similar clauses were interpreted in the past. "
            "Always ground your final decision strictly on the provided Target Contract Text."
        )
        
        # Reconstruct Conversation with formatting
        messages = [{"role": "system", "content": system_message}]
        
        # Turn 1: Setup Contract & First Query
        if not self.history:
            contract_snippet = contract_text[:8000] + "\n[... truncated ...]" if len(contract_text) > 8000 else contract_text
            first_user_content = (
                f"### Target Contract:\n{contract_snippet}\n\n"
                f"### Historical Precedents (External RAG Context):\n{context}\n\n"
                f"### User Question:\n{user_prompt}"
            )
            self.history.append({"role": "user", "content": first_user_content})
        else:
            # Subsequent Turns: Only send the follow-up prompt
            self.history.append({"role": "user", "content": user_prompt})
            
        # Append all historical turns
        messages.extend(self.history)
        
        prompt = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = self.tokenizer(prompt, return_tensors="pt", truncation=True, max_length=4096)
        inputs = {k: v.to(self.model.device) for k, v in inputs.items()}
        
        print("[Agent] Generating response...")
        outputs = self.model.generate(**inputs, max_new_tokens=max_new_tokens, temperature=0.1, top_p=0.9, do_sample=True, pad_token_id=self.tokenizer.pad_token_id)
        
        gen_ids = outputs[0][inputs["input_ids"].shape[1]:]
        raw_response = self.tokenizer.decode(gen_ids, skip_special_tokens=True)
        
        # Save Agent's response to history
        self.history.append({"role": "assistant", "content": raw_response})
        return raw_response

### Run Full Pipeline End-to-End
Combines the Vector/Graph retriever with the loaded local model on sample content to demonstrate the Stateful conversation layer.

In [ ]:
import json

def run_main(contract_path: str, retriever_mode: str, initial_prompt: str):
    """
    Main Orchestrator combining:
    1. Contract Loading
    2. Request Routing (Vector vs Graph)
    3. Generating Response & History Stateful Loop
    """
    with open(contract_path, 'r', encoding='utf-8') as f:
        contract_data = json.load(f)
        current_contract_text = contract_data['documents'][0]['text']

    print(f"Loaded contract from {contract_path} (Length: {len(current_contract_text)} chars)")
    
    if retriever_mode == "vector":
        retriever = VectorRetriever(index_path="/kaggle/working/vector_db")
    elif retriever_mode == "graph":
        retriever = GraphRetriever(graph_path="/kaggle/working/graph_db/contractnli_graph.pkl")
    else:
        raise ValueError("Invalid retriever_mode: Must be vector or graph")
        
    extracted_contexts = retriever.retrieve(initial_prompt)
    flattened_context = "\n".join([chunk["text"] for chunk in extracted_contexts])

    agent = ConversationalAgent(model_identifier="Qwen/Qwen2.5-3B-Instruct")
    result = agent.generate_response(
        user_prompt=initial_prompt, 
        contract_text=current_contract_text, 
        context=flattened_context
    )
    
    print("\n" + "="*50)
    print("--- Conversation Turn 1 ---")
    print(f"User: {initial_prompt}")
    print(f"AI: {result}")
    
    # Provide follow up context
    print("\n--- Conversation Turn 2 (Testing Memory) ---")
    follow_up = "Based on your answer, should I clarify anything?"
    result_2 = agent.generate_response(
        user_prompt=follow_up, 
        contract_text=current_contract_text, 
        context="" # Extracted contexts are maintained in memory loop setup
    )
    
    print(f"User: {follow_up}")
    print(f"AI: {result_2}")
    
# Test End to End on Kaggle using a sample user query targeting non-disclosure rules
run_main("sample.json", "vector", "Does this contract permit me to disclose information to my employees?")